# Reshape data

`data/processed/importaciones_2019-2026.parquet` stores one row per **(declaration item, tax concept)** pair: every shipment-level field (importer, quantity, FOB, countries, etc.) repeats identically across all tax-concept (`COD`) rows belonging to the same declaration, with only `MONTO` varying. This inflates what should be roughly 30M genuine declarations into 672.8M rows.

This step reshapes the raw dataset to one row per declaration item, summing `MONTO` across all tax-concept rows into a single total tax amount column. The per-concept breakdown (`COD`) is not kept.

Reshaping runs before decoding (0.4) on purpose: the AFIP code columns (`ADU`, `M`, `UN`, `DIV`, `PAI`, `PAI_duplicated_0`) are still raw here, but decoding is a deterministic code -> description mapping, so grouping on the raw codes gives the same groups as grouping on their decoded values. Doing it in this order means 0.4's catalog joins only run once per declaration instead of once per (declaration, tax concept) row.

Grouping across the full 672.8M-row raw dataset in one pass exceeds available memory (the same limit that used to make 0.4's joins crash), so each year is reshaped separately into its own parquet file, combined into the final output, then deleted. `FECHA_` is part of the group key, so a declaration's rows never span two years and chunking by year doesn't change the result.


In [1]:
import logging
from pathlib import Path

import polars as pl

logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger(__name__)

processed_dir = Path("../data/processed")
source_path = processed_dir / "importaciones_2019-2026.parquet"
output_path = processed_dir / "importaciones_2019-2026_reshaped.parquet"

# Grouping across the full 672.8M-row raw dataset in one pass exceeds available
# memory, so each year is reshaped separately into its own parquet file,
# combined into the final output, then deleted.
START_YEAR = 2019
END_YEAR = 2026

# Every raw column except COD and MONTO (the tax-concept pair) identifies a
# declaration item; grouping on these collapses the tax-concept rows into one
# row per declaration.
KEY_COLUMNS = [
    "ADU",
    "DESTINACION",
    "NUM_ITEM",
    "FECHA_",
    "NOMBRE_IMPORTADOR",
    "M",
    "UN",
    "CANT_UNIDAD_MEDIDA",
    "FOB_DOLAR",
    "FOB_TOTAL",
    "DIV",
    "PAI",
    "PAI_duplicated_0",
    "POS_NCM",
]


def sum_tax_amounts(lf: pl.LazyFrame, key_columns: list[str]) -> pl.LazyFrame:
    """Collapse tax-concept rows into one row per declaration item.

    Args:
        lf: Lazy frame with one row per (declaration item, tax concept) pair.
        key_columns: Columns that together identify a single declaration item.

    Returns:
        Lazy frame with one row per declaration item and a single
        MONTO_TRIBUTADO_TOTAL column holding the summed tax amount.
    """
    # MONTO is a padded string in the raw data; empty strings (no tax charged)
    # become null before casting so they don't fail the cast.
    lf = lf.with_columns(pl.col("MONTO").replace("", None).cast(pl.Float64))
    return lf.group_by(key_columns).agg(
        pl.col("MONTO").sum().alias("MONTO_TRIBUTADO_TOTAL")
    )


def build_reshaped_lazyframe(year: int) -> pl.LazyFrame:
    """Build the lazy reshape pipeline for a single year of import data.

    Args:
        year: Calendar year to filter the FECHA_ (YYYYMM) column to.

    Returns:
        A LazyFrame with tax-concept rows collapsed, for that year only.
    """
    lf = pl.scan_parquet(source_path).filter(
        pl.col("FECHA_").str.starts_with(str(year))
    )
    return sum_tax_amounts(lf, KEY_COLUMNS)


def reshape_year_to_parquet(year: int, processed_dir: Path) -> Path:
    """Reshape one year of import data into its own parquet file.

    Skips the reshape if the year's file already exists, so re-running the cell
    after a partial failure doesn't repeat already-finished years.

    Args:
        year: Calendar year to reshape.
        processed_dir: Directory to write the year's parquet file under.

    Returns:
        Path to the year's reshaped parquet file.
    """
    year_path = processed_dir / f"importaciones_{year}_reshaped.parquet"
    if not year_path.exists():
        build_reshaped_lazyframe(year).sink_parquet(year_path)
    return year_path


# Reshape each year separately (avoids grouping the full 672.8M-row dataset at
# once, which exceeds available memory), combine the yearly files into the
# final output with a plain, join-free concatenation (safe at full scale,
# since each year's rows are already collapsed to unique declarations), then
# delete the per-year files now that their data lives in the combined output.
yearly_paths = []
for year in range(START_YEAR, END_YEAR + 1):
    logger.info(f"Reshaping {year}...")
    yearly_paths.append(reshape_year_to_parquet(year, processed_dir))

logger.info(f"Combining {len(yearly_paths)} yearly files into {output_path}...")
pl.scan_parquet([str(p) for p in yearly_paths]).sink_parquet(output_path)

for year_path in yearly_paths:
    year_path.unlink()

logger.info(f"Wrote reshaped dataset to {output_path}")


Reshaping 2019...
Reshaping 2020...
Reshaping 2021...
Reshaping 2022...
Reshaping 2023...
Reshaping 2024...
Reshaping 2025...
Reshaping 2026...
Combining 8 yearly files into ..\data\processed\importaciones_2019-2026_reshaped.parquet...
Wrote reshaped dataset to ..\data\processed\importaciones_2019-2026_reshaped.parquet
